{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🏠 ANÁLISE DO MERCADO IMOBILIÁRIO BRASILEIRO\n",
    "## FASE 1: TRATAMENTO DE DADOS E ANÁLISE EXPLORATÓRIA\n",
    "\n",
    "**Dataset:** 10 estados + DF | 2015-2025 | 20+ indicadores\n",
    "\n",
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📋 Índice do Notebook\n",
    "\n",
    "1. [Carregamento dos Dados](#carregamento)\n",
    "2. [Análise Inicial](#analise-inicial)\n",
    "3. [Tratamento de Dados](#tratamento)\n",
    "4. [Criação de Variáveis Derivadas](#variaveis-derivadas)\n",
    "5. [Análise Exploratória Avançada](#exploratoria)\n",
    "6. [Salvando Dados Tratados](#salvar)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 1. Carregamento dos Dados {#carregamento}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Importar bibliotecas\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from datetime import datetime\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Configurar estilo\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 10)\n",
    "\n",
    "print(\"✅ Bibliotecas carregadas com sucesso!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Carregar dados brutos\n",
    "df = pd.read_csv('../data/dados_imobiliarios_brasil_10estados_2015_2025.csv')\n",
    "\n",
    "print(f\"📊 Dataset carregado: {len(df)} registros, {len(df.columns)} colunas\")\n",
    "print(f\"📅 Período: {df['data'].min()} a {df['data'].max()}\")\n",
    "print(f\"📍 Estados: {df['uf'].nunique()}\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 2. Análise Inicial {#analise-inicial}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Informações gerais do dataset\n",
    "df.info()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Estatísticas descritivas\n",
    "df.describe().round(2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Verificar dados faltantes\n",
    "missing = df.isnull().sum()\n",
    "missing_percent = (missing / len(df)) * 100\n",
    "missing_df = pd.DataFrame({\n",
    "    'Coluna': missing.index,\n",
    "    'Valores Faltantes': missing.values,\n",
    "    'Percentual': missing_percent.values\n",
    "})\n",
    "missing_df[missing_df['Valores Faltantes'] > 0]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Verificar distribuição por governo e UF\n",
    "print(\"\\n📊 Distribuição por Governo:\")\n",
    "print(df['governo'].value_counts())\n",
    "\n",
    "print(\"\\n📊 Distribuição por UF:\")\n",
    "print(df['uf'].value_counts())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 3. Tratamento de Dados {#tratamento}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Converter data para datetime\n",
    "df['data'] = pd.to_datetime(df['data'])\n",
    "\n",
    "# Extrair componentes de data\n",
    "df['ano'] = df['data'].dt.year\n",
    "df['mes'] = df['data'].dt.month\n",
    "df['trimestre'] = df['data'].dt.quarter\n",
    "df['semestre'] = df['data'].dt.month.apply(lambda x: 1 if x <= 6 else 2)\n",
    "\n",
    "print(\"✅ Componentes de data adicionados\")\n",
    "df[['data', 'ano', 'mes', 'trimestre', 'semestre']].head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 4. Criação de Variáveis Derivadas {#variaveis-derivadas}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 1. Índice de Crise (0-1)\n",
    "df['indice_crise'] = (df['inadimplencia_aluguel_%'] / 25) + (1 - df['tx_ocupacao_%'] / 100)\n",
    "df['indice_crise'] = df['indice_crise'] / 2\n",
    "\n",
    "# 2. Variação de preço (ano anterior)\n",
    "df = df.sort_values(['uf', 'data'])\n",
    "df['preco_ano_anterior'] = df.groupby('uf')['preco_medio_imovel'].shift(12)\n",
    "df['variacao_preco_anual'] = (df['preco_medio_imovel'] / df['preco_ano_anterior'] - 1) * 100\n",
    "\n",
    "# 3. Abandono total\n",
    "df['abandono_total_por_10k'] = df['abandono_violencia_por_10k'] + \\\n",
    "                               df['abandono_desastre_por_10k'] + \\\n",
    "                               df['abandono_vontade_por_10k']\n",
    "\n",
    "# 4. Rotação de imóveis\n",
    "df['rotacao_imoveis'] = df['vendas_totais_10k_hab'] + df['pessoas_alugam_proprio_10k']\n",
    "\n",
    "# 5. Score de saúde do mercado (0-100)\n",
    "df['saude_mercado'] = 100 - (df['inadimplencia_aluguel_%'] * 2) - \\\n",
    "                      (df['abandono_total_por_10k'] * 3) + \\\n",
    "                      (df['tx_ocupacao_%'] * 0.5) + \\\n",
    "                      (df['qualidade_vida_score'] * 5)\n",
    "df['saude_mercado'] = df['saude_mercado'].clip(0, 100)\n",
    "\n",
    "print(\"✅ Variáveis derivadas criadas:\")\n",
    "df[['indice_crise', 'variacao_preco_anual', 'abandono_total_por_10k', \n",
    "   'rotacao_imoveis', 'saude_mercado']].head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 5. Análise Exploratória Avançada {#exploratoria}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Análise de outliers\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "\n",
    "colunas = ['preco_medio_imovel', 'inadimplencia_aluguel_%', \n",
    "           'vendas_totais_10k_hab', 'qualidade_vida_score']\n",
    "\n",
    "for ax, col in zip(axes.flatten(), colunas):\n",
    "    sns.boxplot(data=df, x='uf', y=col, ax=ax)\n",
    "    ax.set_title(f'Boxplot de {col}', fontweight='bold')\n",
    "    ax.set_xlabel('UF')\n",
    "    ax.set_ylabel(col)\n",
    "    ax.tick_params(axis='x', rotation=45)\n",
    "\n",
    "plt.suptitle('Análise de Outliers por UF', fontsize=16, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Resumo estatístico por UF\n",
    "resumo_uf = df.groupby('uf').agg({\n",
    "    'preco_medio_imovel': ['mean', 'std', 'min', 'max'],\n",
    "    'inadimplencia_aluguel_%': ['mean', 'std'],\n",
    "    'qualidade_vida_score': ['mean', 'std'],\n",
    "    'saude_mercado': ['mean', 'std']\n",
    "}).round(2)\n",
    "\n",
    "resumo_uf = resumo_uf.sort_values(('preco_medio_imovel', 'mean'), ascending=False)\n",
    "resumo_uf"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 6. Salvando Dados Tratados {#salvar}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Selecionar colunas organizadas\n",
    "colunas_ordenadas = [\n",
    "    'data', 'ano', 'mes', 'trimestre', 'semestre',\n",
    "    'uf', 'regiao', 'governo',\n",
    "    'preco_medio_imovel', 'variacao_preco_anual',\n",
    "    'tx_ocupacao_%', 'inadimplencia_aluguel_%',\n",
    "    'vendas_totais_10k_hab', 'rotacao_imoveis',\n",
    "    'perc_vista_%', 'perc_financiamento_%', 'perc_mcmv_%',\n",
    "    'perc_imovel_usado_%', 'perc_terreno_zero_%',\n",
    "    'indice_aluguel_yoy_%',\n",
    "    'qualidade_vida_score', 'indice_crise', 'saude_mercado',\n",
    "    'abandono_violencia_por_10k', 'abandono_desastre_por_10k',\n",
    "    'abandono_vontade_por_10k', 'abandono_total_por_10k',\n",
    "    'aumento_moradores_por_10k', 'invasao_terrenos_por_10k',\n",
    "    'pessoas_alugam_proprio_10k', 'pessoas_vendem_proprio_10k'\n",
    "]\n",
    "\n",
    "df_tratado = df[colunas_ordenadas]\n",
    "\n",
    "# Salvar\n",
    "df_tratado.to_csv('../outputs/dados_tratados.csv', index=False)\n",
    "\n",
    "print(f\"✅ Dados tratados salvos: {len(df_tratado)} registros\")\n",
    "print(f\"📁 Arquivo: outputs/dados_tratados.csv\")\n",
    "df_tratado.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 📊 Resumo da Fase 1\n",
    "\n",
    "✅ **Tratamento concluído com sucesso!**\n",
    "\n",
    "**O que foi feito:**\n",
    "- ✓ Conversão de tipos de dados\n",
    "- ✓ Extração de componentes temporais\n",
    "- ✓ Criação de 5 variáveis derivadas\n",
    "- ✓ Análise de outliers identificada\n",
    "- ✓ Dados salvos em formato limpo\n",
    "\n",
    "**Próximo passo:** Executar `02_analise_estatistica.ipynb`"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}